In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 

import yaml
from pytorch_lightning import Trainer, seed_everything
from corpus.swc_mono_test import SWCMonoTestSetH5Dataset

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

%matplotlib inline
import matplotlib.pyplot as plt

In [2]:

config_path = "config/binaural_attn/word_task_v10_backbone_word_config.yaml"
# config_path = "config/binaural_attn/word_task_half_co_loc_v06.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 96
config['audio']['rep_kwargs']['rep_on_gpu'] = True
# config['getting_acts'] = True 
print(f"Default lr is {config['hparas']['lr']}")
config['hparas']['lr'] = 0.0001
config['model']['attn_constraints']['any'] = True
# print(f"Trying lr = {config['hparas']['lr']}")



Default lr is 0.0001


### Dev arhcitecutre for learned gains 

In [3]:
import torch 
import torch.nn as nn

from src.spatial_attn_architecture import SimpleAttentionalGain, BackBoneCNN

class BackBoneLearnedGains(nn.Module):
    def __init__(self, backbone_config):
        super(BackBoneLearnedGains, self).__init__()
        self.backbone_config = backbone_config
        # init and freeze backbone 
        self.backbone = BackBoneCNN(**backbone_config).eval()
        # freeze backbone 
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone_layers = self.backbone.model_dict
        self.n_layers = len(backbone_config['out_channels'])
        self.gains = nn.ModuleDict()
        for layer in range(self.n_layers):
            # gains placed before each convolutional layer
            self.gains[f'attn{layer}'] = SimpleAttentionalGain(None, None) 
        # add gain between last convolution and fully connected layer 
        self.gains['attnfc'] = SimpleAttentionalGain(None, None)

    def forward(self, cue, mixture, cue_mask_ixs=None):
        cue = self.backbone_layers['norm_coch_rep'](cue)
        attn = self.backbone_layers['attn_coch_rep'](mixture)
        for layer in range(self.n_layers):
            # apply gains
            attn = self.gains[f'attn{layer}'](cue, attn, cue_mask_ixs)
            # get reps from next layer
            cue = self.backbone_layers[f'conv{layer}'](cue)
            attn = self.backbone_layers[f'conv{layer}'](attn)
        # apply gains before fc layer
        out = self.gains['attnfc'](cue, attn, cue_mask_ixs)
        out = out.view(out.size(0), self.backbone.output_size)
        out = self.backbone.fullyconnected(out)        
        out = self.backbone.relufc(out)
        out = self.backbone.dropout(out) 
        return self.backbone.classification(out)

            



In [4]:
# model = BackBoneLearnedGains(config['model'])
### Make sure backbone is not trainable

# backbone_trainable_params = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
# print(backbone_trainable_params)

# gain_trainable_params = sum(p.numel() for p in model.gains.parameters() if p.requires_grad)
# print(gain_trainable_params)


## Try lightning module 



In [5]:
config['model']

{'input_sr': 10000,
 'input_channels': 2,
 'out_channels': [32, 64, 256, 512, 512, 512, 512],
 'kernel': [[2, 34], [2, 14], [5, 5], [5, 5], [6, 6], [5, 5], [6, 6]],
 'stride': [[1, 1], [1, 1], [1, 1], [1, 1], [1, 1], [1, 1], [1, 1]],
 'padding': ['valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time'],
 'pool_stride': [[2, 4], [2, 4], [1, 5], [1, 4], [1, 1], [1, 1], [2, 4]],
 'pool_size': [[9, 13], [9, 13], [1, 13], [1, 13], [1, 1], [1, 1], [6, 13]],
 'pool_padding': [[4, 6], [4, 6], [0, 6], [0, 6], [0, 0], [0, 0], [3, 6]],
 'attn': [0, 0, 0, 0, 0, 0, 0],
 'num_classes': {'num_words': 800},
 'fc_size': 512,
 'global_avg_cue': False,
 'dropout': 0.5,
 'v08': True,
 'attn_constraints': {'any': True},
 'norm_first': True,
 'ln_affine': True,
 'backbone_arch': True}

In [ ]:
import importlib
import src.spatial_attn_lightning as binaural_lightning 
importlib.reload(binaural_lightning)

BinauralAttentionModule = getattr(binaural_lightning, 'BinauralAttentionModule')

# config['model']['backbone_with_learned_gains'] = True 

config_path = "config/binaural_attn/word_task_v10_backbone_learned_gains.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 2
config['hparas']['batch_size'] = 80
config['hparas']['lr'] = 0.001

ckpt_path = "attn_cue_models/word_task_v10_backbone_word_config/checkpoints/epoch=3-step=6227.ckpt"


# module = BinauralAttentionModule.load_from_checkpoint(ckpt_path, config=config)
module = BinauralAttentionModule(config)
state_dict = torch.load(ckpt_path)['state_dict']

## update state dict keys to match new model
new_state_dict = {}
for key, param in state_dict.items():
    new_key = key.replace('_orig_mod.', '_orig_mod.backbone.')
    new_state_dict[key] = param
    new_state_dict[new_key] = param

print(new_state_dict.keys())
module.load_state_dict(new_state_dict, strict=False)

Using explicit dim specification for demeaning in audio transforms
Batch in dataloader = False
Using BackBoneWithLearnedGains
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv -> ReLU


Using dataset BinauralAttentionDataset
OptimizedModule(
  (_orig_mod): BackBoneLearnedGains(
    (backbone): BackBoneCNN(
      (model_dict): ModuleDict(
        (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
        (conv_block_0): Sequential(
          (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
          (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
          (2): ReLU()
        )
        (hann_pool_0): HannPooling2d()
        (conv_block_1): Sequential(
          (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
          (1): Conv2d(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
          (2): ReLU()
        )
        (hann_pool_1): HannPooling2d()
        (conv_block_2): Sequential(
          (0): LayerNorm((64, 10, 1245), eps=1e-05, elementwise_affine=True)
          (1): Conv2d(64, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 0), bias=False)
          (2): ReLU()
       

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


dict_keys(['model._orig_mod.model_dict.norm_coch_rep.weight', 'model._orig_mod.backbone.model_dict.norm_coch_rep.weight', 'model._orig_mod.model_dict.norm_coch_rep.bias', 'model._orig_mod.backbone.model_dict.norm_coch_rep.bias', 'model._orig_mod.model_dict.conv_block_0.0.weight', 'model._orig_mod.backbone.model_dict.conv_block_0.0.weight', 'model._orig_mod.model_dict.conv_block_0.0.bias', 'model._orig_mod.backbone.model_dict.conv_block_0.0.bias', 'model._orig_mod.model_dict.conv_block_0.1.weight', 'model._orig_mod.backbone.model_dict.conv_block_0.1.weight', 'model._orig_mod.model_dict.hann_pool_0.hann_window2d', 'model._orig_mod.backbone.model_dict.hann_pool_0.hann_window2d', 'model._orig_mod.model_dict.conv_block_1.0.weight', 'model._orig_mod.backbone.model_dict.conv_block_1.0.weight', 'model._orig_mod.model_dict.conv_block_1.0.bias', 'model._orig_mod.backbone.model_dict.conv_block_1.0.bias', 'model._orig_mod.model_dict.conv_block_1.1.weight', 'model._orig_mod.backbone.model_dict.conv

_IncompatibleKeys(missing_keys=['model._orig_mod.backbone.model_dict.attn0.bias', 'model._orig_mod.backbone.model_dict.attn0.slope', 'model._orig_mod.backbone.model_dict.attn0.threshold', 'model._orig_mod.backbone.model_dict.attn1.bias', 'model._orig_mod.backbone.model_dict.attn1.slope', 'model._orig_mod.backbone.model_dict.attn1.threshold', 'model._orig_mod.backbone.model_dict.attn2.bias', 'model._orig_mod.backbone.model_dict.attn2.slope', 'model._orig_mod.backbone.model_dict.attn2.threshold', 'model._orig_mod.backbone.model_dict.attn3.bias', 'model._orig_mod.backbone.model_dict.attn3.slope', 'model._orig_mod.backbone.model_dict.attn3.threshold', 'model._orig_mod.backbone.model_dict.attn4.bias', 'model._orig_mod.backbone.model_dict.attn4.slope', 'model._orig_mod.backbone.model_dict.attn4.threshold', 'model._orig_mod.backbone.model_dict.attn5.bias', 'model._orig_mod.backbone.model_dict.attn5.slope', 'model._orig_mod.backbone.model_dict.attn5.threshold', 'model._orig_mod.backbone.model_

Process ForkProcess-2:
Process ForkProcess-1:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/concurrent/futures/process.py", line 244, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/py

In [7]:
### Make sure pl module correctly counts what is / is not trainable

backbone_trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)
print(backbone_trainable_params)


24


In [8]:
### import pl trainer
from pytorch_lightning import Trainer

trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    gradient_clip_val=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model            | OptimizedModule      

Using v06 dataset
/om/scratch/Fri/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v10
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
568 files in train concat dataset
len training set = 35216
Epoch 0:   0%|          | 0/35216 [00:00<?, ?it/s] 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:77: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 80. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 0:   0%|          | 67/35216 [05:29<48:02:13,  0.20it/s, v_num=4.11e+7, train_loss_step=5.980, grad_norm=1.320]

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
